# user websockets
> Solveit compatible user-defined HTMX websockets.

In [ ]:
#| default_exp user_ws

In [ ]:
#| export
import json, os
from fastcore.foundation import store_attr
from fastcore.meta import delegates
from pyskills.core import allow
from dialoghelper import add_scr_a, iife_a, js_eval_a
from pote.dutil import wait_for_a
from pote.basic import id_gen


In [ ]:
import os, json
from fastcore.foundation import patch, AttrDict as AD
from fastcore.test import *
from pyskills import doc
from pyskills import *
from fasthtml.core import FastHTML
from fasthtml.components import Div, Span
from dialoghelper.core import add_msg, add_html_a, read_msg, add_scr_a, js_run_a
from pote.logger import logger
from pote.dutil import link_msg
from pote.server import stop_server, ensure_server, local_server, is_serving

In [ ]:
#| export
_new_id = id_gen()

## Problem

Solveit injects browser JavaScript that listens globally for HTMX websocket lifecycle events:

```javascript
function updateWSIndicator(type) {
  if (['error', 'close'].includes(type)) {
    htmx.find('#submit_btn').disabled = true;
    htmx.find('#cancel_btn').disabled = true;
  }
  const color = { error: "orange", open: "limegreen", close: "red" }[type] || 'grey';
  const elt = document.getElementById("ws-indicator");
  if (elt) elt.style.backgroundColor = color;
}

htmx.on("htmx:wsOpen", ({ detail: { event} }) => updateWSIndicator(event.type));
htmx.on("htmx:wsClose", ({ detail: { event } }) => updateWSIndicator(event.type));
htmx.on("htmx:wsError", () => updateWSIndicator('error'));
```

That works for Solveit’s own websocket, but it is **too broad**. If a user also uses HTMX’s websocket extension in their own app inside a dialog, their websocket emits the same `htmx:wsOpen`, `htmx:wsClose`, and `htmx:wsError` events.

So when the user’s own websocket server stops, Solveit sees `htmx:wsClose` / `htmx:wsError` and thinks **Solveit’s own backend connection** is broken. The result is:

- Solveit’s WS indicator turns red/orange
- Submit/cancel buttons may be disabled
- Console logs misleading messages like “Lost connection to solveit”
- HTMX may keep retrying the user websocket using Solveit’s global `htmx.config.wsReconnectDelay`

The real problem is not that Solveit’s websocket is down. The user’s websocket is down. Solveit is reacting to the wrong websocket events.

## Important environment detail

Inside Solveit, relative HTTP requests like:

```python
Span(hx_get='/foo', hx_target='closest div')('private')
```

can work because the Solveit backend proxies unhandled HTTP requests to `localhost:8000`.

But that proxy does **not** forward websocket upgrades. So this will not work reliably:

```html
<div hx-ext="ws" ws-connect="/ws2"></div>
```

because `/ws2` goes through Solveit’s own app/proxy path.

For user websockets, use the public `.solveit.pub` domain mapped to the user server port:

```python
ws_url = f"wss://{json.loads(os.environ['PUBLIC_DOMAINS'])[str(srv.port)]}/ws2"
```

Then inject the websocket initiator using that absolute `wss://...` URL:

```python
await add_html_a(Div(Div(ws_connect=ws_url, hx_ext='ws', id='ws2-div'), hx_swap_oob='afterend:#ephemeral'))
```

The `.solveit.pub` mapping supports websocket upgrade; verified with `101 Switching Protocols`.

## Why CORS is not the main issue

Browser WebSocket connections are not blocked by normal fetch/XHR CORS rules. The browser sends an `Origin` header, but it is up to the server to accept or reject it. If the public `.solveit.pub` tunnel forwards the upgrade correctly, the websocket should connect.

## First workaround: capture and block user websocket events

HTMX’s `htmx.on('event', fn)` attaches the listener to `document.body`, not `document`. This matters.

If we add a capturing-phase event listener on `document`, it runs before Solveit’s bubbling-phase listener on `document.body`.

So we can stop user websocket events before Solveit sees them:

```javascript
document.addEventListener('htmx:wsOpen', e => {
  if (e.target?.id !== 'ws2-div') return;
  window._myWs = e.detail.socketWrapper;
  e.stopImmediatePropagation();
  console.log('non-solveit WS opened');
}, true)

document.addEventListener('htmx:wsClose', e => {
  if (e.target?.id !== 'ws2-div') return;
  e.stopImmediatePropagation();
  console.log('blocked close from my ws');
}, true)

document.addEventListener('htmx:wsError', e => {
  if (e.target?.id !== 'ws2-div') return;
  e.stopImmediatePropagation();
  console.log('blocked error from my ws');
}, true)
```

Key details:

- Use `e.target?.id === 'ws2-div'`, not `socketWrapper` comparison.
- `htmx:wsError` uses a different wrapper shape than `htmx:wsOpen` / `htmx:wsClose`, so wrapper identity comparison is fragile.
- Target-element comparison survives reconnects, since each retry still targets the same websocket initiator element.
- This prevents Solveit’s indicator/listeners from reacting to the user websocket.

This fixes the false red/orange Solveit indicator.

## Remaining issue: HTMX still retries

Stopping propagation does not stop HTMX’s internal websocket retry logic. The retry is scheduled inside the websocket extension’s own `onclose` / `onerror` handlers before or independently of the DOM event propagation.

So even if Solveit’s UI no longer reacts, the console may still show repeated failed websocket reconnect attempts.

For intentional shutdown, the clean solution is:

```javascript
const el = document.getElementById('ws2-div')
if (el) el.remove()
```

Then stop the Python server:

```python
stop_server()
```

Removing the websocket initiator element lets HTMX clean up the websocket and avoids the retry loop. In testing, this also avoided triggering the close/error blockers at all.

So the practical shutdown sequence is:

```python
await iife_a(r'''
const el = document.getElementById('ws2-div')
if (el) el.remove()
''')

stop_server()
```

## Best overall pattern

Use both:

1. Capturing listeners as a safety net for unexpected disconnects
2. Remove the websocket initiator element before intentional server shutdown

The listeners prevent Solveit’s UI from reacting to user websocket failures. Removing the initiator avoids HTMX’s retry loop when shutdown is intentional.

## Browser → Python websocket messages

The FastHTML websocket handler expects structured JSON matching the handler parameters. Sending plain text caused a server parse error:

```javascript
window._myWs.send('world')
```

That produced:

```text
Expecting value: line 1 column 1 (char 0)
```

Correct browser send:

```javascript
window._myWs.send(JSON.stringify({msg: 'world'}))
```

Corresponding Python handler:

```python
@app.ws('/ws2', conn=on_connect, disconn=on_disconnect)
async def ws2(msg:str, send):
    _msgs.append(msg)
    if msg == '__init__': return
    await send(Div('Hello ' + msg, id='note2'))
```

## Python → browser websocket messages

FastHTML gives a `send` callable on websocket connect. Store it, then use it later from any async Python cell:

```python
_msgs = []
_ws_send = None

async def on_connect(send):
    global _ws_send
    _ws_send = send
    _msgs.append('Connected!')

async def on_disconnect(ws):
    global _ws_send
    _ws_send = None
    _msgs.append('Disconnected!')
```

Then later:

```python
await _ws_send(Div('Hello from Python', id='note2'))
```

That sends server → browser over the already-open websocket.

## Subtle timing issue with `on_connect`

Sending UI HTML from `on_connect` is unreliable.

Observed behavior:

- Server sent: `<div id="note2">Hello, you have connected</div>`
- Browser websocket inspector showed the message was received
- But HTMX did not swap it into the DOM
- Later messages, sent after explicit browser messages, swapped correctly

Adding `hx-swap-oob="true"` did not fix the first-message issue. A real `setTimeout` delay after `htmx:wsOpen` did work, suggesting a timing/settling race: the websocket is open and messages arrive, but HTMX/Solveit may not yet be ready to process the first swap.

Best practice: treat `conn=on_connect` as lifecycle bookkeeping only. Don’t send UI updates from it.

Good shape:

```python
async def on_connect(send):
    global _ws_send
    _ws_send = send
    _msgs.append('Connected!')

async def on_disconnect(ws):
    global _ws_send
    _ws_send = None
    _msgs.append('Disconnected!')

@app.ws('/ws2', conn=on_connect, disconn=on_disconnect)
async def ws2(msg:str, send):
    _msgs.append(msg)
    if msg == '__init__': return
    await send(Div('Hello ' + msg, id='note2'))
```

## Minimal complete browser-side workaround

```javascript
document.addEventListener('htmx:wsOpen', e => {
  if (e.target?.id !== 'ws2-div') return;
  window._myWs = e.detail.socketWrapper;
  e.stopImmediatePropagation();
  console.log('non-solveit WS opened');
}, true)

document.addEventListener('htmx:wsClose', e => {
  if (e.target?.id !== 'ws2-div') return;
  e.stopImmediatePropagation();
  console.log('blocked close from my ws');
}, true)

document.addEventListener('htmx:wsError', e => {
  if (e.target?.id !== 'ws2-div') return;
  e.stopImmediatePropagation();
  console.log('blocked error from my ws');
}, true)
```

Clean shutdown:

```javascript
const el = document.getElementById('ws2-div')
if (el) el.remove()
```

Browser send:

```javascript
window._myWs.send(JSON.stringify({msg: 'world'}))
```

Python send:

```python
await _ws_send(Div('Hello from Python', id='note2'))
```

## Bottom line

The Solveit bug/oversight is that its websocket monitor listens to all HTMX websocket events globally and assumes they belong to Solveit. User HTMX websockets produce the same events, so user websocket failures are misinterpreted as Solveit connection failures.

The workaround is to:

- Connect user websockets via the public `wss://...solveit.pub/...` URL, not relative paths
- Give the user websocket initiator a stable id, e.g. `ws2-div`
- Add capturing-phase document listeners to block `htmx:wsOpen`, `htmx:wsClose`, and `htmx:wsError` from that element
- Remove the websocket initiator before intentionally stopping the user server
- Avoid sending DOM updates from `on_connect`; send only after the websocket is fully settled or after an explicit client message

## version 0

###### start-section

In [ ]:
# app, rt, srv, p = ensure_server(app=FastHTML(exts='ws', debug=True, 
#     surreal=False, htmx=False, default_hdrs=False))
# 
# _msgs = []
# _ws_send = None
# 
# async def on_connect(send):
#     global _ws_send
#     _ws_send = send
#     print(send)
#     _msgs.append('Connected!')
# 
# async def on_disconnect(ws):
#     global _ws_send
#     _ws_send = None
#     print(ws, dir(ws), ws.send)
#     _msgs.append('Disconnected!')
# 
# @app.ws('/ws2', conn=on_connect, disconn=on_disconnect)
# async def ws2(msg:str, send):
#     _msgs.append(msg)
#     if msg == '__init__': return
#     await send(Div('Hello ' + msg, id="note2"))

In [ ]:
# await iife_a('''
# document.addEventListener('htmx:wsOpen', e => {
#   if (e.target?.id !== 'ws2-div') return;
#   window._myWs = e.detail.socketWrapper;
#   e.stopImmediatePropagation();
#   console.log('blocked open from my ws');
#   setTimeout(() => window._myWs.send(JSON.stringify({msg: '__init__'})), 100)
# }, true)
# 
# document.addEventListener('htmx:wsClose', (e) => {
#   if (e.target?.id === 'ws2-div') { e.stopImmediatePropagation(); console.log('blocked close from my ws') }
# }, true)
# 
# document.addEventListener('htmx:wsError', (e) => {
#   if (e.target?.id === 'ws2-div') { e.stopImmediatePropagation(); console.log('blocked error from my ws') }
# }, true)
# 
# htmx.on('htmx:wsClose', () => console.log('solveit saw wsClose'))
# htmx.on('htmx:wsError', () => console.log('solveit saw wsError'))
# ''')

In [ ]:
# ws_url = f"wss://{json.loads(os.environ['PUBLIC_DOMAINS'])[str(srv.port)]}/ws2"
# await add_html_a(Div(Div(ws_connect=ws_url, hx_ext='ws', id='ws2-div'), hx_swap_oob='afterend:#ephemeral'))

In [ ]:
# while '__init__' not in _msgs: await asyncio.sleep(0.05)
# _msgs

```javascript
window._myWs.send(JSON.stringify({msg: 'world'}))
```

In [ ]:
# await _ws_send(Div('Hello from Python', id='note2'))

In [ ]:
# _msgs

In [ ]:
# res = await js_eval_a('''
# const el = document.getElementById('ws2-div')
# if (el) { el.remove(); return 'removed'; } else { return 'not found'; }
# ''')
# print(res)
# stop_server()

In [ ]:
_user_ws_js = r'''

'''
#| linkedto: _5e8e7990

###### end-section

In [ ]:
_ = '\n'.join((await read_msg()).content.splitlines()[1:])
await link_msg(f"_user_ws_js = r'''\n{_}\n'''", msg_type='code');

In [ ]:
_user_ws_js = r'''
debugger;
if (window.user_ws) {
  window.user_ws.listen(false);
  delete window.user_ws;
}
window.user_ws ??= (function() {
  'use strict'
  const ids = new Set();

  // function _match(e) { return ids.has(e.target?.id) }
  function _updateWSIndicator(type) {
    const color = { error: "orange", open: "limegreen", close: "red" }[type] || 'grey';
    $('#uws-indicator').css('backgroundColor', color)
  }
  function _onopen(e) {
    if (!e.target.hasAttribute('data-uws')) return;
    e.target._ws = e.detail.socketWrapper;
    ids.add(e.target.id);
    setTimeout(() => send(e.target.id, {msg: '__init__'}), 250);
    _updateWSIndicator(e.detail.event.type);
    e.stopImmediatePropagation();
  }
  function _onerr(e) {
    if (!e.target.hasAttribute('data-uws')) return;
    _updateWSIndicator(e.detail.event.type);
    e.stopImmediatePropagation();
  }
  function listen(on=true) {
    const l = (on?document.addEventListener:document.removeEventListener).bind(document);
    l('htmx:wsOpen', _onopen, true);
    l('htmx:wsClose', _onerr, true);
    l('htmx:wsError', _onerr, true);
  }
  
  // function create(id, wsUrl) {
  //   const el = document.createElement('div');
  //   el.id = id;
  //   el.setAttribute('hx-ext', 'ws');
  //   el.setAttribute('ws-connect', wsUrl);
  //   document.body.appendChild(el);
  //   ids.add(id);
  //   if (window.htmx) htmx.process(el);
  //   _updateWSIndicator()
  //   return 'created';
  // }

  function cleanup(id) {
    const el = document.getElementById(id);
    if (el) {
      el.remove();
      ids.delete(id);
      return 'removed';
    } else { return 'not found'; }
  }

  function send(id, data) {
    const el = document.getElementById(id);
    if (el?._ws) el._ws.send(typeof data === 'string' ? data : JSON.stringify(data));
  }
  
  listen();
  return { listen, cleanup, send };
})();
'''
#| linkedto: _f2d7c7c4

In [ ]:
async def setup_user_ws():
    return await add_scr_a(_user_ws_js)

In [ ]:
# Div(Div(cls='flex items-center gap-1 min-w-0 flex-3/5')(
#     Span('uws: '), Span(id='uws-indicator---', cls='size-4 border rounded-full min-w-4'),
# ),
# Div(id='uws-note')('Waiting...')
# )

In [ ]:
# await add_html_a(Div(hx_swap_oob='afterend:#ws-indicator')(Span(id='uws-indicator', cls='size-4 border rounded-full min-w-4')));
await add_html_a(
    Div(hx_swap_oob='beforebegin:#dialog-container')(
        Div(id='uws-box', cls='flex flex-wrap w-full gap-1 lg:flex-nowrap')(
            Span(id='uws-indicator', cls='size-4 border rounded-full min-w-4', title='wid:'),
            Div(id='uws-note')('Waiting...'),
        )
    )
);

In [ ]:
await setup_user_ws()

We're going to refactor the exploration in an re-usable python/JS library.

Look at the above javascript source. Key design decisions:
- **`Set` of IDs** instead of a `Map` — simpler, since the socket wrapper can live on the element itself (`el._ws`)
- **Each handler references `e.target` directly** — no shared `el` variable
- **Filtering by element ID is sufficient**
- **`el._ws`** stores the socket wrapper for later access (e.g. `document.getElementById('ws2-div')._ws.send(...)`)

One thing to consider: should `setup` also create/inject the DOM element, or just register the ID for the listeners to watch? Currently it only registers — the element injection happens separately (via `add_html_a` in the Python code). Perhaps both paths, having another function that `setup` calls optionally so the user can opt to inject the html explicitly or create the initiator from javascript?


##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good design observations. On the `setup` question — I'd lean toward keeping `setup` as registration-only, and adding a separate `create(id, wsUrl)` that builds the `<div>` with the right `hx-ext`, `ws-connect`, and `id` attributes, appends it to the DOM, and calls `setup(id)` internally. That way:

- **Python-driven path**: `add_html_a(...)` injects the element from Python, then `setup(id)` registers it
- **JS-driven path**: `create(id, wsUrl)` does both in one call

Single responsibility for `setup`, and users who already have their element injected (the current workflow) don't need to change anything.

The alternative — having `setup` optionally create the element — would mean conditionally building DOM, which muddies the contract. A separate `create` keeps each function's behavior predictable.

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Looking at where we are: the JS module is injected and working. The next logical step is a Python-side helper to construct the `wss://` URL from the public domains env var, since that's needed every time we connect a user websocket:

```python
def ws_url(srv, path='/ws2'):
    return f"wss://{json.loads(os.environ['PUBLIC_DOMAINS'])[str(srv.port)]}{path}"
```

Run that and we can build the next piece on top — a Python function that registers the WS route, injects the DOM element via `user_ws.create`, and wires up `on_connect`/`on_disconnect` bookkeeping.

<details class='token-usage-details' markdown='1'><summary>$0.0900</summary>

`total=41,605 | in=41,010 | out=595 | cached=0.0% | $0.0900 | glm-5p2`

</details>


In [ ]:
def ws_url(port=8000, path='/ws2'):
    return f"wss://{json.loads(os.environ['PUBLIC_DOMAINS'])[str(port)]}{path}"

In [ ]:
async def user_ws_create(wid, wsurl):
    await add_html_a(Div(Div(ws_connect=wsurl, hx_ext='ws', id=wid, data_uws=True), hx_swap_oob='afterend:#ws-div'))
async def user_ws_cleanup(wid):
    return await js_eval_a(f"return user_ws.cleanup({json.dumps(wid)})")

In [ ]:
ws_url(path='/ws2?abc=ABC')

'wss://known-vent-santa-eras34.solveit.pub/ws2?abc=ABC'

In [ ]:
app, rt, srv, p = ensure_server(app=FastHTML(exts='ws', debug=True, surreal=False, htmx=False, default_hdrs=False))#, log_level='info')

In [ ]:
_msgs = []
_ws_send = None

async def on_connect(send):
    global _ws_send
    _ws_send = send
    _msgs.append('Connected!')

async def on_disconnect(ws):
    global _ws_send
    _ws_send = None
    _msgs.append('Disconnected!')

@app.ws('/ws2', conn=on_connect, disconn=on_disconnect)
async def ws2(wid:str, msg:str, send):
    _msgs.append((wid, msg))
    if msg == '__init__': return
    await send(Div(f"Hello {msg} from {wid}", id='note2'))

In [ ]:
wid = 'ws2-div'
res = await js_eval_a(f"return user_ws.cleanup('{wid}')")
test_eq(res['result'], 'not found')
await user_ws_create(wid, ws_url(path=f"/ws2?wid={wid}"))

In [ ]:
await wait_for_a(lambda: _ws_send is not None)
await wait_for_a(lambda: '__init__' in str(_msgs))
_msgs

['Connected!', ('ws2-div', '__init__')]

Console Websockets mesages:
```
{"msg":"__init__"}
```

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great — that means the good path is working now:

- `htmx:wsOpen` fired
- our capture listener matched `ws2-div`
- it blocked Solveit from seeing the event
- `el._ws` was set
- the delayed `__init__` was sent through the HTMX socket
- Python received the structured message

Next smallest check is to verify Python saw it:

```python
_msgs
```

<details class='token-usage-details' markdown='1'><summary>$0.0071</summary>

`total=46,660 | in=46,567 | out=93 | cached=14.0% | $0.0071 | gpt-5.5`

</details>

In [ ]:
await iife_a("user_ws.send('ws2-div', {msg: 'world'})")
await wait_for_a(lambda: 'world' in str(_msgs[-1]))

res = await user_ws_cleanup('ws2-div')
res, _msgs

({'data_id': '7555b95e-0a2a-4560-9455-4c35abe4f143', 'result': 'removed'},
 ['Connected!', ('ws2-div', '__init__'), ('ws2-div', 'world')])

In [ ]:
_msgs.clear()
print(await js_eval_a("return user_ws.cleanup('ws2-div')"))
print(await user_ws_create('ws3-div', ws_url(path=f"/ws2?wid=ws3-div")))
await wait_for_a(lambda: _ws_send is not None)
await wait_for_a(lambda: '__init__' in str(_msgs))
await _ws_send(Div('Hello from Python', id='note2'))
_msgs

{'data_id': '19dec967-a1e6-4327-8061-c7cb1f78436b', 'result': 'not found'}


None


['Connected!', ('ws3-div', '__init__')]

websockets messages:
```
{"msg":"__init__"}	18	11:49:57.800
<div id="note2">Hello from Python</div>	40	11:59:57.474
```
I see `Hello from Python` output.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great — that verifies the full round trip:

- browser → Python works through `user_ws.send`
- Python receives structured JSON as `msg`
- Python → browser works through `_ws_send`
- HTMX swaps the returned `<div id="note2">...</div>` into the page

The next useful refactor target is to stop relying on a single global `_ws_send`, since multiple websocket initiators can connect to the same endpoint. A small registry keyed by websocket id / route will be more reusable.

<details class='token-usage-details' markdown='1'><summary>$0.0072</summary>

`total=47,501 | in=47,394 | out=107 | cached=13.8% | $0.0072 | gpt-5.5`

</details>


In [ ]:
res = await user_ws_cleanup('ws3-div')
test_eq(res['result'], 'removed')

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Next small step: replace the single `_ws_send` with a tiny registry, while keeping the current `/ws2` handler shape.

```python
_ws_sends = {}

def ws_conn(id):
    async def _conn(send): _ws_sends[id] = send
    return _conn

def ws_disconn(id):
    async def _disconn(ws): _ws_sends.pop(id, None)
    return _disconn
```

Then a tiny helper for Python → browser sends:

```python
async def ws_send(id, o): return await _ws_sends[id](o)
```

<details class='token-usage-details' markdown='1'><summary>$0.0072</summary>

`total=47,665 | in=47,537 | out=128 | cached=13.7% | $0.0072 | gpt-5.5`

</details>


In [ ]:
_ws_sends = {}

def ws_conn(wid):
    async def _conn(send): _ws_sends[wid] = send
    return _conn

def ws_disconn(wid):
    async def _disconn(ws): _ws_sends.pop(wid, None)
    return _disconn

In [ ]:
async def ws_send(wid, o): return await _ws_sends[wid](o)

In [ ]:
wid = 'ws4-div'
@app.ws('/ws4', conn=ws_conn(wid), disconn=ws_disconn(wid))
async def ws4(wid:str, msg:str, send):
    _msgs.append((wid, msg))
    if msg == '__init__': return
    await send(Div(f"Hello {msg} from {wid}", id='note2'))

In [ ]:
await user_ws_create(wid, ws_url(path=f"/ws4?wid={wid}"))
await wait_for_a(lambda: wid in _ws_sends)
_ws_sends.keys()

dict_keys(['ws4-div'])

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good. Now verify browser → Python and Python → browser on the registry-backed socket:

```python
await iife_a("user_ws.send('ws4-div', {msg: 'registry'})")
await wait_for_a(lambda: 'registry' in _msgs)
await ws_send('ws4-div', Div('Hello from registry', id='note2'))
```

<details class='token-usage-details' markdown='1'><summary>$0.0073</summary>

`total=48,380 | in=48,303 | out=77 | cached=13.5% | $0.0073 | gpt-5.5`

</details>


In [ ]:
await iife_a("user_ws.send('ws4-div', {msg: 'registry'})")
await wait_for_a(lambda: 'registry' in str(_msgs[-1]))
await ws_send(wid, Div('Hello from registry', id='note2'))

```
{"msg":"registry"}	18	12:03:38.200
{"msg":"__init____"}	18	12:04:06.446
<div id="note2">Hello registry</div>	37	12:04:06.562
<div id="note2">Hello from registry</div>	42	12:04:06.571
```

`#note2` updated.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Excellent — the registry-backed version is verified in both directions.

Next small refactor: make a helper that both registers the route and returns the initiator id/path pair, so callers don’t have to repeat the id in three places.

```python
def add_user_ws(app, id, path, handler, target='note2'):
    app.ws(path, conn=ws_conn(id), disconn=ws_disconn(id))(handler)
    return id,path
```

A tiny test endpoint can use it like this:

```python
async def ws5(msg:str, send):
    _msgs.append(msg)
    if msg == '__init__': return
    await send(Div('Hello ' + msg, id='note2'))

add_user_ws(app, 'ws5-div', '/ws5', ws5)
```

<details class='token-usage-details' markdown='1'><summary>$0.0074</summary>

`total=48,792 | in=48,621 | out=171 | cached=5.0% | $0.0074 | gpt-5.5`

</details>


In [ ]:
def add_user_ws(app, wid, path, handler):
    app.ws(path, conn=ws_conn(wid), disconn=ws_disconn(wid))(handler)

In [ ]:
async def ws5(wid:str, msg:str, send):
    _msgs.append((wid, msg))
    if msg == '__init__': return
    await send(Div(f"Hello {msg} from {wid}", id='note2'))

wid, path = 'ws5-div', '/ws5'
add_user_ws(app, wid, path, ws5)

In [ ]:
await user_ws_create(wid, ws_url(path=path+f"?wid={wid}"))

<div class="prose" markdown="1">

```python
{'data_id': '1edcb6f2-a926-4d0d-a315-94a838b177fb', 'result': 'created'}
```

</div>

In [ ]:
await wait_for_a(lambda: wid in _ws_sends)
await iife_a("user_ws.send('ws5-div', {msg: 'helper'})")
await wait_for_a(lambda: 'helper' in str(_msgs[-1]))
await ws_send(wid, Div('Hello from helper', id='note2'))

In [ ]:
async def start_user_ws(srv, wid, path): return await user_ws_create(wid, ws_url(srv.port, path+f"?wid={wid}"))

In [ ]:
wid, path = 'ws6-div', '/ws6'
add_user_ws(app, wid, path, ws5)
await start_user_ws(srv, wid, path)
await wait_for_a(lambda: wid in _ws_sends)
await iife_a("user_ws.send('ws6-div', {msg: 'started'})")
await wait_for_a(lambda: 'started' in str(_msgs[-1]))
await ws_send(wid, Div('Hello from start_user_ws', id='note2'))

All good.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Nice. We now have the pieces separated cleanly:

- `ws_url` builds the public websocket URL
- `user_ws_create` creates/processes the browser initiator
- `_ws_sends` tracks active Python send callables by initiator id
- `ws_conn` / `ws_disconn` manage that registry
- `ws_send` sends Python → browser
- `add_user_ws` registers an endpoint
- `start_user_ws` starts the browser side

The next worthwhile step is packaging these into a small reusable section, probably with the JS string plus these Python helpers together.

<details class='token-usage-details' markdown='1'><summary>$0.0076</summary>

`total=50,013 | in=49,886 | out=127 | cached=95.2% | $0.0076 | gpt-5.5`

</details>


In [ ]:
await user_ws_cleanup('ws4-div')
await user_ws_cleanup('ws5-div')
await user_ws_cleanup('ws6-div');

## version 1

Let's consolidate the Python-facing helper layer first:

In [ ]:
def user_ws_url(srv, path, qs=''): return f"wss://{json.loads(os.environ['PUBLIC_DOMAINS'])[str(srv.port)]}{path}{qs}"

async def user_ws_call(fn, *args):
    args = ', '.join(json.dumps(o) for o in args)
    return await js_eval_a(f"return user_ws.{fn}({args})")

async def user_ws_create(wid, wsurl): return await user_ws_call('create', wid, wsurl)
async def user_ws_cleanup(wid): return await user_ws_call('cleanup', wid)
async def user_ws_send(wid, data): return await iife_a(f"user_ws.send({json.dumps(wid)}, {json.dumps(data)})")

In [ ]:
wid,path = 'ws7-div', '/ws7'
add_user_ws(app, wid, path, ws5)
await user_ws_create(wid, user_ws_url(srv, path, f"?wid={wid}"))
await wait_for_a(lambda: wid in _ws_sends)
await user_ws_send(wid, dict(msg='packed'))
await wait_for_a(lambda: 'packed' in str(_msgs[-1]))
await ws_send(wid, Div('Hello from packed helpers', id='note2'))

await user_ws_cleanup(wid)

<div class="prose" markdown="1">

```python
{'data_id': 'f86b4ab1-7bf9-45d4-b0ca-5d070cea64f2', 'result': 'removed'}
```

</div>

Consolidate the server-side registry helpers into the same reusable shape, with a clearer public name:

In [ ]:
user_ws_sends = {}

def user_ws_conn(wid):
    async def _conn(send): user_ws_sends[wid] = send
    return _conn

def user_ws_disconn(wid):
    async def _disconn(ws): user_ws_sends.pop(wid, None)
    return _disconn

async def user_ws_send_py(wid, o): return await user_ws_sends[wid](o)

def add_user_ws(app, wid, path, handler):
    app.ws(path, conn=user_ws_conn(wid), disconn=user_ws_disconn(wid))(handler)
    return wid,path

In [ ]:
wid,path = 'ws8-div', '/ws8'
add_user_ws(app, wid, path, ws5)
await user_ws_create(wid, user_ws_url(srv, path, f"?wid={wid}"))
await wait_for_a(lambda: wid in user_ws_sends)
await user_ws_send(wid, dict(msg='renamed'))
await wait_for_a(lambda: 'renamed' in str(_msgs[-1]))
await user_ws_send_py(wid, Div('Hello from renamed registry', id='note2'))
await user_ws_cleanup(wid)

<div class="prose" markdown="1">

```python
{'data_id': '8651088a-e568-45ef-ab00-07c61657e125', 'result': 'removed'}
```

</div>

Make a single convenience function that registers the route and starts the browser initiator.

In [ ]:
async def start_user_ws(app, srv, wid, path, handler):
    add_user_ws(app, wid, path, handler)
    return await user_ws_create(wid, user_ws_url(srv, path, f"?wid={wid}"))

In [ ]:
wid, path = 'ws9-div', '/ws9'
await start_user_ws(app, srv, wid, path, ws5)
await wait_for_a(lambda: wid in user_ws_sends)
await user_ws_send(wid, dict(msg='onecall'))
await wait_for_a(lambda: 'onecall' in str(_msgs[-1]))
await user_ws_send_py(wid, Div('Hello from start_user_ws', id='note2'))

Make shutdown symmetric so intentional shutdown removes the initiator first and then drops the stored send callable.

In [ ]:
async def stop_user_ws(id):
    try: await user_ws_cleanup(id)
    finally: user_ws_sends.pop(id, None)

In [ ]:
await stop_user_ws(wid)
test_is(wid in user_ws_sends, False)

Add a tiny readiness helper so callers don't repeat `wait_for_a(lambda: id in user_ws_sends)`.

In [ ]:
async def wait_user_ws(wid): return await wait_for_a(lambda: wid in user_ws_sends)

In [ ]:
wid, path = 'ws10-div', '/ws10'
await start_user_ws(app, srv, wid, path, ws5)
await wait_user_ws(wid)
test_is(wid in user_ws_sends, True)

Make `start_user_ws` optionally wait until the socket is registered before returning. That gives callers a safer default.

In [ ]:
async def start_user_ws(app, srv, wid, path, handler, wait=True):
    res = add_user_ws(app, wid, path, handler)
    await user_ws_create(wid, user_ws_url(srv, path, f"?wid={wid}"))
    if wait: await wait_user_ws(wid)
    return res

In [ ]:
wid, path = 'ws11-div', '/ws11'
await start_user_ws(app, srv, wid, path, ws5)
test_is(wid in user_ws_sends, True)

In [ ]:
await stop_user_ws('ws10-div')
await stop_user_ws('ws11-div')

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good. Next small packaging step is to collect the helper names intended for export, so it’s clear what the public surface is:

```python
_user_ws_exports = 'setup_user_ws user_ws_url user_ws_call user_ws_create user_ws_cleanup user_ws_send user_ws_sends user_ws_conn user_ws_disconn user_ws_send_py add_user_ws start_user_ws stop_user_ws wait_user_ws'.split()
_user_ws_exports
```

<details class='token-usage-details' markdown='1'><summary>$0.0081</summary>

`total=53,465 | in=53,374 | out=91 | cached=98.6% | $0.0081 | gpt-5.5`

</details>


That's a lot and a lot of `user_ws_` prefixes.  Time to put in a class, don't you think?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes — this is the right point to put the public surface behind one object. The class can own `app`, `srv`, JS namespace, and the active send registry.

In [ ]:
class UserWS:
    def __init__(self, app, srv, js='user_ws'):
        store_attr()
        self.sends = {}

    def url(self, path, qs=''): return f"wss://{json.loads(os.environ['PUBLIC_DOMAINS'])[str(self.srv.port)]}{path}{qs}"

    async def call(self, fn, *args):
        args = ', '.join(json.dumps(o) for o in args)
        return await js_eval_a(f"return {self.js}.{fn}({args})")

    async def create(self, wid, path):
        wsurl = path if path.startswith('ws') else self.url(path, f"?wid={wid}")
        return await self.call('create', wid, wsurl)

    async def cleanup(self, wid): return await self.call('cleanup', wid)
    async def send_client(self, wid, data): return await iife_a(f"{self.js}.send({json.dumps(wid)}, {json.dumps(data)})")
    async def send(self, wid, o):
        assert wid in self.sends, f"{wid} is not connected"
        return await self.sends[wid](o)
    async def wait(self, wid): return await wait_for_a(lambda: wid in self.sends)

    def conn(self, wid):
        async def _conn(send): self.sends[wid] = send
        return _conn

    def disconn(self, wid):
        async def _disconn(ws): self.sends.pop(wid, None)
        return _disconn

    def add(self, path, handler, wid=''):
        wid = wid or _new_id('uws')
        self.app.ws(path, conn=self.conn(wid), disconn=self.disconn(wid))(handler)
        return wid

    async def start(self, path, handler, wid='', wait=True):
        wid = self.add(path, handler, wid=wid)
        await self.create(wid, path)
        if wait: await self.wait(wid)
        return wid
    async def restart(self, path, handler, wid, wait=True): return await self.start(path, handler, wid=wid, wait=wait)

    async def stop(self, wid):
        try: await self.cleanup(wid)
        finally: self.sends.pop(wid, None)

In [ ]:
uws = UserWS(app, srv)

In [ ]:
wid = await uws.start(path := '/ws12', ws5)
test_is(wid in uws.sends, True)
wid

'uws_1'

In [ ]:
await uws.send_client(wid, dict(msg='classy'))
await wait_for_a(lambda: 'classy' in str(_msgs[-1]))
await uws.send(wid, Div('Hello from UserWS', id='note2'))

All good.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great. The class shape is now verified:

- `uws.start(...)` registers the endpoint, creates/processes the browser initiator, and waits for connection
- `uws.send_client(...)` sends browser → Python
- `uws.send(...)` sends Python → browser
- `uws.sends` cleanly scopes active send callables per `UserWS` instance

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Next useful cleanup: make `start` return `wid,path` as before, but first clean up any existing initiator with the same id to avoid duplicate DOM elements/retry loops during iteration.

In [ ]:
@patch
async def start(self:UserWS, path, handler, wid='', wait=True):
    if wid: await self.stop(wid)
    wid = self.add(path, handler, wid)
    await self.create(wid, path)
    if wait: await self.wait(wid)
    return wid

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

One thing to be careful about: `FastHTML.app.ws(...)` may not like re-registering the same route path during iteration. Cleaning up the DOM initiator is good, but `self.add(...)` still registers the route again.

So next tiny refinement: split “route already added” from “browser initiator started” by tracking registered websocket ids.

In [ ]:
@patch
def _init_routes(self:UserWS):
    if not hasattr(self, 'routes'): self.routes = set()

@patch
def add(self:UserWS, path, handler, wid=''):
    self._init_routes()
    wid = wid or _new_id('uws')
    if wid not in self.routes:
        self.app.ws(path, conn=self.conn(wid), disconn=self.disconn(wid))(handler)
        self.routes.add(wid)
    return wid

In [ ]:
await uws.start(path, ws5, wid)
test_eq((wid in uws.sends, wid in uws.routes), (True, True))

In [ ]:
await uws.send_client(wid, dict(msg='restarted'))
await wait_for_a(lambda: 'restarted' in str(_msgs[-1]))
await uws.send(wid, Div('Hello after restart', id='note2'))

All good.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Nice. One more small polish: initialize `routes` in `__init__` so `_init_routes` isn't needed in the final class.

In [ ]:
@patch
def __init__(self:UserWS, app, srv, js='user_ws'):
    store_attr()
    self.sends,self.routes = {},set()

@patch
def add(self:UserWS, path, handler, wid=''):
    wid = wid or _new_id('uws')
    if wid not in self.routes:
        self.app.ws(path, conn=self.conn(wid), disconn=self.disconn(wid))(handler)
        self.routes.add(wid)
    return wid

In [ ]:
uws2 = UserWS(app, srv)
wid2 = await uws2.start(path2 := '/ws13', ws5, wid)
test_eq((wid2 == wid, wid2 in uws2.sends, wid2 in uws2.routes), (True, True, True))

In [ ]:
await uws2.send_client(wid2, dict(msg='fresh_instance'))
await wait_for_a(lambda: 'fresh_instance' in str(_msgs[-1]))
await uws2.send(wid2, Div('Hello from fresh instance', id='note2'))

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Next small tightening: `routes` should probably be a dict from `wid → path`, not a set. That makes restart idempotent for the same `wid/path`, but catches accidental reuse of the same `wid` for a different route.

```python
@patch
def __init__(self:UserWS, app, srv, js='user_ws'):
    store_attr()
    self.sends,self.routes = {},{}

@patch
def add(self:UserWS, path, handler, wid=''):
    wid = wid or _new_id('uws')
    if wid in self.routes:
        assert self.routes[wid] == path, f"{wid} already registered for {self.routes[wid]}, not {path}"
        return wid
    self.app.ws(path, conn=self.conn(wid), disconn=self.disconn(wid))(handler)
    self.routes[wid] = path
    return wid
```

<details class='token-usage-details' markdown='1'><summary>$0.0074</summary>

`total=46,295 | in=45,616 | out=679 | cached=14.3% | reasoning=483 | $0.0074 | gpt-5.5`

</details>


In [ ]:
@patch
def __init__(self:UserWS, app, srv, js='user_ws'):
    store_attr()
    self.sends,self.routes = {},{}

@patch
def add(self:UserWS, path, handler, wid=''):
    wid = wid or _new_id('uws')
    if wid in self.routes:
        assert self.routes[wid] == path, f"{wid} already registered for {self.routes[wid]}, not {path}"
        return wid
    self.app.ws(path, conn=self.conn(wid), disconn=self.disconn(wid))(handler)
    self.routes[wid] = path
    return wid

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Tiny test for the duplicate-`wid` guard:

```python
uws3 = UserWS(app, srv)
wid3 = uws3.add('/ws14', ws5, 'same-wid')
test_eq(uws3.add('/ws14', ws5, 'same-wid'), wid3)
test_fail(lambda: uws3.add('/ws15', ws5, 'same-wid'), contains='already registered')
```

<details class='token-usage-details' markdown='1'><summary>$0.0070</summary>

`total=46,115 | in=46,020 | out=95 | cached=5.3% | $0.0070 | gpt-5.5`

</details>


In [ ]:
uws3 = UserWS(app, srv)
wid3 = uws3.add('/ws14', ws5, 'same-wid')
test_eq(uws3.add('/ws14', ws5, 'same-wid'), wid3)
test_fail(lambda: uws3.add('/ws15', ws5, 'same-wid'), contains='already registered')

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Next useful polish: make `stop` tolerate unknown ids and return the browser cleanup result, since that’s handy in notebooks/tests.

```python
@patch
async def stop(self:UserWS, wid):
    try: return await self.cleanup(wid)
    finally: self.sends.pop(wid, None)
```

<details class='token-usage-details' markdown='1'><summary>$0.0070</summary>

`total=46,337 | in=46,268 | out=69 | cached=14.1% | $0.0070 | gpt-5.5`

</details>


In [ ]:
@patch
async def stop(self:UserWS, wid):
    try: return await self.cleanup(wid)
    finally: self.sends.pop(wid, None)

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Test the revised `stop` behavior with both existing and unknown ids:

```python
wid4 = await uws3.start('/ws14', ws5, 'same-wid')
test_eq(wid4 in uws3.sends, True)
res = await uws3.stop(wid4)
test_eq(res['result'], 'removed')
test_eq(wid4 in uws3.sends, False)
res = await uws3.stop('missing-wid')
test_eq(res['result'], 'not found')
```

<details class='token-usage-details' markdown='1'><summary>$0.0070</summary>

`total=46,540 | in=46,428 | out=112 | cached=5.2% | $0.0070 | gpt-5.5`

</details>


In [ ]:
wid4 = await uws3.start('/ws14', ws5, 'same-wid')
test_eq(wid4 in uws3.sends, True)
res = await uws3.stop(wid4)
test_eq(res['result'], 'removed')
test_eq(wid4 in uws3.sends, False)
res = await uws3.stop('missing-wid')
test_eq(res['result'], 'not found')

In [ ]:
await uws2.stop(wid2)
stop_server()

## version 2

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good. The class works from a fresh instance.

Next, let's prepare the final consolidated JS string and class in two clean cells. First the JS string:

In [ ]:
_ = '\n'.join((await read_msg()).content.splitlines()[1:])
await link_msg(f"_user_ws_js = r'''\n{_}\n'''", msg_type='code', is_exported=True);

In [ ]:
#| export
_user_ws_js = r'''
__debugger__if (window.user_ws) {
  window.user_ws.listen(false);
  delete window.user_ws;
}
window.user_ws = (function() {
  'use strict'
  const ids = new Set();

  function _match(e) { return ids.has(e.target?.id) }
  function _onopen(e) {
    if (!_match(e)) return;
    const sw = e.detail.socketWrapper;
    e.target._ws = sw;
    setTimeout(() => send(e.target.id, {event:'__init__', wid:e.target.id}), 250);
    e.stopImmediatePropagation();
  }
  function _onerr(e) {
    if (!_match(e)) return;
    e.stopImmediatePropagation();
  }
  function listen(on=true) {
    const l = (on?document.addEventListener:document.removeEventListener).bind(document);
    l('htmx:wsOpen', _onopen, true);
    l('htmx:wsClose', _onerr, true);
    l('htmx:wsError', _onerr, true);
  }
  function create(id, wsUrl) {
    const el = document.createElement('div');
    el.id = id;
    el.setAttribute('hx-ext', 'ws');
    el.setAttribute('ws-connect', wsUrl);
    document.body.appendChild(el);
    ids.add(id);
    if (window.htmx) htmx.process(el);
    return 'created';
  }
  function cleanup(id) {
    const el = document.getElementById(id);
    if (el) {
      el.remove();
      ids.delete(id);
      return 'removed';
    } else return 'not found';
  }
  function send(id, data) {
    const el = document.getElementById(id);
    if (el?._ws) el._ws.send(JSON.stringify(data));
  }
  listen();
  const doc = 'Browser helper for Solveit-safe HTMX websocket initiators';
  return {doc, listen, create, cleanup, send};
})();
'''
#| linkedto: _99847640

In [ ]:
#| export
async def setup_user_ws(debugger=False):
    "Inject the browser-side HTMX websocket event filter and initiator helper"
    return await add_scr_a(_user_ws_js.replace('__debugger__', 'debugger;\n' if debugger else ''))

In [ ]:
await setup_user_ws(True)

#| export
The browser helper in `_user_ws_js` is injected once with `setup_user_ws()`, while `UserWS(app, srv)` instances can be created many times to manage independent route/send registries.

The JS helper tracks websocket initiator element IDs in a `Set`, stores each HTMX socket wrapper on its DOM element as `el._ws`, and filters HTMX websocket events by `e.target.id`. Matching `htmx:wsOpen`, `htmx:wsClose`, and `htmx:wsError` events are stopped in the capture phase so Solveit’s own global websocket monitor does not mistake user websocket lifecycle events for Solveit backend failures.

In [ ]:
#| export
class UserWS:
    "Manage many Solveit-safe HTMX websocket initiators and server send handles keyed by `wid`"
    def __init__(self, app, srv): store_attr(); self.sends,self.routes = {},set()

    def url(self, path):
        "Return public websocket URL for `path`"
        return f"wss://{json.loads(os.environ['PUBLIC_DOMAINS'])[str(self.srv.port)]}{path}"

    _js='user_ws'
    async def call(self, fn, *args):
        "Call a function on the browser-side websocket helper"
        args = ', '.join(json.dumps(o) for o in args)
        return await js_eval_a(f"return {self._js}.{fn}({args})")

    async def create(self, wid, path):
        "Create browser websocket initiator for `path`"
        return await self.call('create', wid, self.url(path))
    async def cleanup(self, wid):
        "Remove browser initiator element"
        return await self.call('cleanup', wid)
    async def send_client(self, wid, data):
        "Send browser to Python via JS helper"
        return await iife_a(f"{self._js}.send({json.dumps(wid)}, {json.dumps(data)})")
    async def send(self, wid, o):
        "Send Python to browser via stored send callable"
        return await self.sends[wid](o)
    async def wait(self, wid):
        "Wait for websocket connection to be registered"
        return await wait_for_a(lambda: wid in self.sends)

    def _conn(self, wid, conn):
        "Return FastHTML connect hook storing server send callable"
        async def _conn(send):
            self.sends[wid] = send
            if conn: await conn(send)
        return _conn

    def _disconn(self, wid, disconn):
        "Return FastHTML disconnect hook clearing server send callable"
        async def _disconn(ws):
            self.sends.pop(wid, None)
            if disconn: await disconn(ws)
        return _disconn

    def add(self, path, handler, wid='', conn=None, disconn=None, **kwargs
    ) -> str: # `wid`
        "Register websocket route once; optionally setup conn/disconn hooks"
        wid = wid or _new_id('uws', '_')
        if wid not in self.routes:
            self.app.ws(path, conn=self._conn(wid, conn), disconn=self._disconn(wid, disconn), **kwargs)(handler)
            self.routes.add(wid)
        return wid

    @delegates(add, keep=True)
    async def start(self, path, handler, wid='', wait=True, **kwargs
    )-> str: # `wid`
        "Register route, create browser initiator, and optionally wait for connection"
        if wid: await self.stop(wid)
        wid = self.add(path, handler, wid, **kwargs)
        await self.create(wid, path)
        if wait: await self.wait(wid)
        return wid

    async def stop(self, wid):
        "Remove browser initiator and forget server send callable"
        try: await self.cleanup(wid)
        finally: self.sends.pop(wid, None)

In [ ]:
# app, rt, srv, p = ensure_server(app=FastHTML(exts='ws', debug=True, surreal=False, htmx=False, default_hdrs=False))

In [ ]:
Div(Div(cls='flex items-center gap-1 min-w-0 flex-3/5')(Span('uws: '),),(lg := Div(id='note3')('Waiting...')))

<div><div class="flex items-center gap-1 min-w-0 flex-3/5"><span>uws: </span></div><div id="note3">Waiting...</div></div>

In [ ]:
# app=FastHTML(exts='ws')
# app=FastHTML(exts='ws', debug=True, surreal=False, htmx=False, default_hdrs=False)
_msgs.clear()

# async with local_server(app=app) as srv:
async with local_server(appkw=AD(exts='ws')) as srv:
    uws3 = UserWS(srv.app, srv)

    async def ws14(abc:str, data:dict, send):
        print(abc, kw)
        _msgs.append(data)
        if '__init__' in str(data): return
        await send(Div('Hello ' + str(data), id='note3'))

    wid3 = await uws3.start('/ws14?abc=ABC', ws14, name='ws14')
    test_eq((wid3 in uws3.sends,wid3 in uws3.routes), (True, True))
    await wait_for_a(lambda: len(_msgs) > 0 and '__init__' in str(_msgs[-1]))

    print(_msgs, '\n----\n')

    await uws3.send_client(wid3, dict(msg='clean'))
    await wait_for_a(lambda: 'clean' in str(_msgs[-1]))
    await uws3.send(wid3, Div('Hello from cleaned UserWS', id='note3'))

    await uws3.stop(wid3)
    print(_msgs)

In [ ]:
await uws3.send_client(wid3, dict(msg='clean'))
await wait_for_a(lambda: 'clean' in str(_msgs[-1]))
await uws3.send(wid3, Div('Hello from cleaned UserWS', id='note3'))

_msgs

In [ ]:
await uws3.stop(wid3)

Perfect.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great — we now have a clean, verified reusable shape:

- browser-side JS owns HTMX event filtering, initiator creation/cleanup, and client sends
- `UserWS` owns route registration, public URL creation, connection registry, lifecycle, and Python sends
- restart/cleanup avoids duplicate DOM initiators and HTMX retry loops
- routes are tracked so restarts don't re-register the same handler unnecessarily

<details class='token-usage-details' markdown='1'><summary>$0.0092</summary>

`total=60,658 | in=60,571 | out=87 | cached=98.7% | $0.0092 | gpt-5.5`

</details>


In [ ]:
uws4 = UserWS(app, srv)

async def ws15(msg:dict, send):
    _msgs.append(msg)
    if msg['event'] == '__init__': return
    await send(Div('Hello ' + msg, id='note2'))

async def _conn(send): print('connected')
async def _disconn(send): print('disconnected')

wid4 = await uws4.start(path4 := '/ws15', ws15, conn=_conn, disconn=_disconn)
await uws4.send_client(wid4, dict(msg='doc_ok'))
await wait_for_a(lambda: 'doc_ok' in _msgs)
await uws4.send(wid4, Div('Hello after doc reinject', id='note2'))

In [ ]:
await uws4.stop(wid4)

#| export
`UserWS` provides Solveit-safe user HTMX websockets. One `UserWS` instance can manage many independent websocket initiators, keyed by `wid`.

Typical flow:

1. `await setup_user_ws()` once per browser session
2. `uws = UserWS(app, srv)`
3. define a FastHTML websocket handler
4. `wid = await uws.start('/ws', handler)`
5. browser → Python: `await uws.send_client(wid, dict(msg='hi'))`
6. Python → browser: `await uws.send(wid, Div('hello', id='target'))`
7. cleanup before stopping the server: `await uws.stop(wid)`

`UserWS.start()` removes any existing browser initiator with the same id before recreating it, avoiding duplicate DOM elements and HTMX retry loops. Routes are registered only once per `UserWS` instance.

Python handlers passed to `UserWS.start()` are normal FastHTML websocket handlers registered with `app.ws(...)`. They receive the usual parsed parameters plus `send`, and can send normal FastHTML responses. FastHTML websocket responses are always treated as OOB swaps. `UserWS` only manages connection setup, event isolation, and access to the server-side `send` callable; it does not change FastHTML websocket handler semantics.

### Environment notes

- Solveit's HTTP proxy does not forward websocket upgrades; always use the public `wss://...solveit.pub/...` URL from `UserWS.url()`, not relative paths.
- `on_connect` has a timing race: the socket is open and messages arrive, but HTMX may not yet process the first swap. Treat `conn=` as lifecycle bookkeeping only; send UI updates after an explicit client message.
- The browser helper sends `{msg: '__init__'}` 250ms after `htmx:wsOpen` to confirm the connection is settled before the handler sees real messages. Handlers should `return` early on `msg == '__init__'`.

**Example**:

```python
async def ws_demo(msg:str, send):
    if msg == '__init__': return
    await send(Div('Hello ' + msg, id='note2'))

async def _conn(send): print('connected')
async def _disconn(send): print('disconnected')

uws = UserWS(app, srv)
wid = await uws.start('/demo-ws', ws_demo, conn=_conn, disconn=_disconn)
await uws.send_client(wid, dict(msg='world'))

await uws.stop(wid)
```

In [ ]:
#| export
allow(setup_user_ws, {UserWS: ...})

In [ ]:
stop_server()

# pyskill

In [ ]:
import pote.user_ws
from pote.user_ws import setup_user_ws, UserWS

In [ ]:
list_pyskills()['pote.user_ws']

In [ ]:
doc(pote.user_ws)

In [ ]:
doc(setup_user_ws)

In [ ]:
doc(UserWS)

# export -

In [ ]:
from pote.flakes import show_flakes
await show_flakes()

<div class="prose" markdown="1">

No warnings to report

</div>

In [ ]:
# #|hide
# #|eval: false
# from pote.dialog import dlg_export
# await dlg_export()